# 13 - 全栈分布式 Trace

**目标**: 整合 09-12 所有层级的测量数据，生成完整的请求链路甘特图，回答「从用户点击到看到第一个字，每毫秒发生了什么」。

**完整事件链**:
```
[UI-1]   用户点击「发送」
[UI-2]   React onClick → setState(loading) → 重渲染 (~3ms)
[UI-3]   new EventSource(url) 建立 SSE 连接
[NET-1]  TCP 握手到 API Gateway (:8001) (~1ms loopback)
[NET-2]  HTTP GET /graphql/stream 请求到达
[API-1]  Nginx 转发开销（生产环境 ~0.5ms）
[API-2]  FastAPI 路由匹配 (~3ms)
[API-3]  CORS 中间件 (~0.1ms)
[API-4]  GraphQL 解析 /graphql/stream 端点 (~0ms, 是 GET 路由)
[API-5]  API Gateway → AI Engine HTTP hop (:8002) (~1ms)
[ENGINE-1] FastAPI 路由匹配 (~3ms)
[ENGINE-2] Session 加载（内存 dict, ~0.1ms）
[AGENT-1] GreeterAgent.route() → call_llm(greeter_prompt)
            LLM TTFB + 生成: ~500-2000ms
[AGENT-2] RequirementAgent → call_llm(requirement_prompt)
            字段提取: ~500-2000ms（JSON响应，非流式）
[AGENT-2b] category_agent.identify_category() (~5ms, 关键词)
[ENGINE-3] SSE first chunk → event_generator() yield
[NET-BACK] SSE 数据包反向传输 (~1ms loopback)
[API-BACK] API Gateway SSE 透传（proxy_buffering=off）
[UI-4]   EventSource.onmessage() 触发
[UI-5]   setState(messages) → React 重渲染 (~3ms)
[UI-6]   DOM 更新 + 浏览器 paint (~2ms)
[USER]   用户看到第一个字
```

**关键指标**:
- **TTFB** (Time To First Byte): API Gateway 接收请求到第一个 SSE 字节
- **FIChar** (First Interactive Character): 用户点击到看到第一个字符
- **E2E** (End-to-End): 完整响应结束

In [ ]:
# ============================================================
# CELL 0: 配置区域 — 填入各 Notebook 的实测数据
# ============================================================

# --- 服务端点 ---
API_GATEWAY_URL = "http://localhost:8001"
AI_ENGINE_URL   = "http://localhost:8002"
FRONTEND_URL    = "http://localhost:3001"

# --- 采集参数 ---
N_SAMPLES = 5     # 全链路 E2E 测试次数（含 LLM，次数少）
TIMEOUT_S = 120.0

# ============================================================
# 手动基线数据（从 09-12 运行结果填入，或设为 None 自动加载）
# ============================================================
AUTO_LOAD = True  # True = 尝试自动读取最新 CSV/JSON；False = 使用手动数据

# 手动数据（AUTO_LOAD=False 时使用，单位 ms）
MANUAL_DATA = {
    # --- 网络层 (Notebook 09) ---
    "net_dns_ms":          0.5,   # DNS 查询 (loopback=0)
    "net_tcp_ms":          0.8,   # TCP 握手
    "net_tls_ms":          0.0,   # TLS (本地无 TLS)
    "net_server_proc_ms":  5.0,   # 服务器处理 (FastAPI 路由)
    "net_download_ms":     0.5,   # 响应下载

    # --- API 网关层 (Notebook 10) ---
    "api_routing_ms":      4.0,   # FastAPI 路由匹配
    "api_cors_ms":         0.3,   # CORS 中间件
    "api_gql_parse_ms":    3.0,   # GraphQL 解析（仅 hello query）
    "api_hop_ms":          1.0,   # 内部 HTTP hop

    # --- Agent 层 (Notebook 11) ---
    "agent_greeter_ms":    1500,  # GreeterAgent LLM
    "agent_requirement_ms": 1800, # RequirementAgent LLM
    "agent_category_ms":   5.0,   # 品类识别（关键词）
    "agent_clerk_ms":      2500,  # ClerkAgent LLM（仅完整流程）
    "agent_session_ms":    0.5,   # Session 读写

    # --- UI 层 (Notebook 12) ---
    "ui_click_to_state_ms": 1.0,  # onClick → setState
    "ui_react_render_ms":   3.0,  # React 重渲染
    "ui_dom_paint_ms":      2.0,  # DOM + 浏览器 paint
    "ui_sse_connect_ms":    8.0,  # EventSource 建立连接
}

# --- 输出路径 ---
OUTPUT_DIR = "../data"

import time as _time
_ts = int(_time.time())

print("配置加载完成")
print(f"  AUTO_LOAD: {AUTO_LOAD}")
print(f"  手动数据: {len(MANUAL_DATA)} 项")

In [ ]:
# ============================================================
# CELL 1: 依赖导入
# ============================================================

import httpx
import json
import os
import time
import urllib.parse
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("依赖加载完成")

In [ ]:
# ============================================================
# CELL 2: 自动加载历史测量数据
# ============================================================

baseline = MANUAL_DATA.copy()

if AUTO_LOAD:
    data_dir = Path(OUTPUT_DIR)
    loaded_count = 0

    # 加载 Notebook 09 (网络层)
    net_jsons = sorted(data_dir.glob("network-layer-deep_*.json"), reverse=True)
    if net_jsons:
        with open(net_jsons[0]) as f:
            net_data = json.load(f)
        # 尝试提取关键指标
        for endpoint_key in ["api_gateway_health", "ai_engine_health"]:
            if endpoint_key in net_data.get("httpx_results", {}):
                d = net_data["httpx_results"][endpoint_key]
                if d.get("ttfb_mean_ms"):
                    baseline["net_server_proc_ms"] = d["ttfb_mean_ms"]
                    loaded_count += 1
                    print(f"  [09] net_server_proc_ms = {d['ttfb_mean_ms']}ms")
                    break

    # 加载 Notebook 10 (API 网关)
    gw_jsons = sorted(data_dir.glob("api-gateway-internals_*.json"), reverse=True)
    if gw_jsons:
        with open(gw_jsons[0]) as f:
            gw_data = json.load(f)
        for key, field in [
            ("routing_results", "API GW /health"),
            ("hop_results",     "_estimated_hop_ms"),
        ]:
            if key in gw_data:
                d = gw_data[key]
                if field in d and isinstance(d[field], dict):
                    if d[field].get("mean"):
                        baseline["api_routing_ms"] = d[field]["mean"]
                        loaded_count += 1
                        print(f"  [10] api_routing_ms = {d[field]['mean']}ms")
                elif field == "_estimated_hop_ms" and isinstance(d.get(field), (int, float)):
                    baseline["api_hop_ms"] = d[field]
                    loaded_count += 1
                    print(f"  [10] api_hop_ms = {d[field]}ms")

    # 加载 Notebook 11 (Agent 层)
    agent_jsons = sorted(data_dir.glob("agent-orchestration-trace_*.json"), reverse=True)
    if agent_jsons:
        with open(agent_jsons[0]) as f:
            agent_data = json.load(f)
        # 从 agent_timeline 加载
        for item in agent_data.get("agent_timeline", []):
            name = item.get("name", "")
            ms   = item.get("ms", 0)
            if "GreeterAgent" in name and ms:
                baseline["agent_greeter_ms"] = ms
                loaded_count += 1
                print(f"  [11] agent_greeter_ms = {ms}ms")
            elif "字段提取" in name and ms:
                baseline["agent_requirement_ms"] = ms
                loaded_count += 1
                print(f"  [11] agent_requirement_ms = {ms}ms")
            elif "Clerk" in name and ms:
                baseline["agent_clerk_ms"] = ms
                loaded_count += 1
                print(f"  [11] agent_clerk_ms = {ms}ms")

    # 加载 Notebook 12 (UI 层)
    ui_jsons = sorted(data_dir.glob("ui-frontend-metrics_*.json"), reverse=True)
    if ui_jsons:
        with open(ui_jsons[0]) as f:
            ui_data = json.load(f)
        sse_results = ui_data.get("sse_results", [])
        if sse_results:
            t_connects = [r["t_connect_ms"] for r in sse_results if r.get("t_connect_ms")]
            if t_connects:
                baseline["ui_sse_connect_ms"] = round(np.mean(t_connects), 2)
                loaded_count += 1
                print(f"  [12] ui_sse_connect_ms = {baseline['ui_sse_connect_ms']}ms")

    print(f"\n  自动加载: {loaded_count} 项指标从历史数据更新")

print("\n当前基线数据:")
for k, v in baseline.items():
    print(f"  {k:<35} = {v} ms")

In [ ]:
# ============================================================
# CELL 3: 全链路 E2E 实测（注入 X-Request-ID）
# 使用唯一 request_id 追踪请求在各层的时间戳
# ============================================================

print("=" * 60)
print("全链路 E2E 实测")
print("=" * 60)

import uuid

e2e_results = []

# 使用流式端点测量 TTFB 和 E2E
TEST_MESSAGES = [
    ("partial",  "我要采购一批电脑"),
    ("full",     "我需要采购100台华硕笔记本电脑，用于办公室员工，预算50万，2026年3月交货，北京总部，i7/16G，无特殊要求"),
]

for msg_type, message in TEST_MESSAGES:
    print(f"\n  场景: {msg_type} ({len(message)} 字)")
    scenario_results = []

    for i in range(N_SAMPLES):
        request_id = str(uuid.uuid4())[:8]
        sid = f"e2e-{msg_type}-{_ts}-{i}"

        t_start = time.perf_counter()
        t_connect  = None  # HTTP 响应头到达
        t_first    = None  # 第一个 content chunk
        t_complete = None  # complete 事件
        chunk_count = 0

        try:
            with httpx.Client(timeout=TIMEOUT_S) as c:
                with c.stream(
                    "POST",
                    f"{AI_ENGINE_URL}/api/v1/requirement/stream",
                    json={"message": message, "session_id": sid},
                    headers={
                        "X-Request-ID": request_id,
                        "Accept": "text/event-stream",
                    }
                ) as resp:
                    t_connect = (time.perf_counter() - t_start) * 1000
                    current_event = None

                    for line in resp.iter_lines():
                        if line.startswith("event:"):
                            current_event = line[6:].strip()
                        elif line.startswith("data:"):
                            try:
                                d = json.loads(line[5:].strip())
                                if d.get("content") and t_first is None:
                                    t_first = (time.perf_counter() - t_start) * 1000
                                chunk_count += 1
                            except:
                                pass
                        elif current_event == "complete" and t_complete is None:
                            t_complete = (time.perf_counter() - t_start) * 1000

            t_total = (time.perf_counter() - t_start) * 1000
            result = {
                "request_id":  request_id,
                "msg_type":    msg_type,
                "t_connect_ms":   t_connect,
                "t_first_ms":     t_first,
                "t_complete_ms":  t_complete,
                "t_total_ms":     t_total,
                "chunk_count":    chunk_count,
            }
            scenario_results.append(result)
            e2e_results.append(result)

            print(f"  [{i+1}/{N_SAMPLES}] id={request_id} "
                  f"connect={t_connect:.0f}ms "
                  f"first={t_first:.0f}ms "
                  f"total={t_total:.0f}ms "
                  f"chunks={chunk_count}")

        except Exception as e:
            print(f"  [{i+1}/{N_SAMPLES}] 错误: {e}")

    # 统计
    if scenario_results:
        t_firsts = [r["t_first_ms"] for r in scenario_results if r["t_first_ms"]]
        t_totals = [r["t_total_ms"] for r in scenario_results]
        print(f"  [统计] {msg_type}: TTFB mean={np.mean(t_firsts):.0f}ms  Total mean={np.mean(t_totals):.0f}ms")

print(f"\n总测量次数: {len(e2e_results)}")

In [ ]:
# ============================================================
# CELL 4: 构建完整链路甘特图数据
# ============================================================

print("=" * 60)
print("构建完整链路甘特图")
print("=" * 60)

# 从 E2E 实测中提取代表性时间
e2e_partial = [r for r in e2e_results if r["msg_type"] == "partial"]
e2e_full    = [r for r in e2e_results if r["msg_type"] == "full"]

def e2e_mean(results, key):
    vals = [r[key] for r in results if r.get(key)]
    return round(np.mean(vals), 0) if vals else None

# 完整采购流程的关键时间节点
t_full_first = e2e_mean(e2e_full, "t_first_ms") or baseline["agent_greeter_ms"] + baseline["agent_requirement_ms"]
t_full_total = e2e_mean(e2e_full, "t_total_ms") or t_full_first + baseline["agent_clerk_ms"]

# 构建甘特图各层事件
# 格式: (层级, 事件名, 开始时间ms, 结束时间ms, 颜色, 详情)
def build_chain(t_greeter, t_req, t_clerk, t_total):
    """
    基于测量值构建事件链。
    t_greeter: GreeterAgent TTFB
    t_req:     RequirementAgent 处理时间
    t_clerk:   ClerkAgent 处理时间（完整流程）
    t_total:   完整 E2E
    """
    b = baseline

    # 时间节点计算（累积）
    t0  = 0   # 用户点击
    t1  = b["ui_click_to_state_ms"]                                    # setState(loading)
    t2  = t1 + b["ui_react_render_ms"]                                 # React 重渲染
    t3  = t2 + 0.5                                                     # EventSource 初始化
    t4  = t3 + b["net_tcp_ms"]                                         # TCP 握手
    t5  = t4 + b["api_routing_ms"]                                     # FastAPI 路由
    t6  = t5 + b["api_cors_ms"]                                        # CORS 中间件
    t7  = t6 + b["api_hop_ms"]                                         # API → Engine hop
    t8  = t7 + 2.0                                                     # Engine FastAPI 路由
    t9  = t8 + b["agent_session_ms"]                                   # Session 加载
    t10 = t9 + t_greeter                                               # GreeterAgent LLM
    t11 = t10 + t_req                                                  # RequirementAgent LLM
    t12 = t11 + b["agent_category_ms"]                                 # 品类识别
    t13 = t12 + b["net_download_ms"] + b["api_hop_ms"]                # SSE 反向传输
    t14 = t13 + 0.5                                                    # onmessage 触发
    t15 = t14 + b["ui_react_render_ms"]                               # React 重渲染
    t16 = t15 + b["ui_dom_paint_ms"]                                   # DOM + paint

    chain = [
        # (tier, name, start, end, color)
        ("UI 层",      "用户点击",              t0,  t1,  "#2ecc71"),
        ("UI 层",      "setState+重渲染",        t1,  t2,  "#3498db"),
        ("UI 层",      "EventSource 初始化",    t3,  t4,  "#9b59b6"),
        ("网络层",     "TCP 握手",              t4,  t5,  "#e67e22"),
        ("API 网关",   "FastAPI 路由+CORS",     t5,  t7,  "#f39c12"),
        ("API 网关",   "→Engine HTTP hop",     t7,  t8,  "#d35400"),
        ("AI Engine",  "Engine 路由+Session",  t8,  t9,  "#c0392b"),
        ("AI Engine",  "GreeterAgent LLM",    t9,  t10, "#e74c3c"),
        ("AI Engine",  "Req. LLM+品类识别",   t10, t12, "#e74c3c"),
        ("网络层",     "SSE 反向传输",          t12, t13, "#e67e22"),
        ("UI 层",      "onmessage+重渲染",      t13, t15, "#3498db"),
        ("UI 层",      "DOM+Paint→首字",       t15, t16, "#2ecc71"),
    ]

    return chain, t16

chain_partial, t_partial_total = build_chain(
    baseline["agent_greeter_ms"],
    baseline["agent_requirement_ms"],
    baseline["agent_clerk_ms"],
    t_full_total
)

print(f"\n完整链路时间节点:")
for tier, name, start, end, color in chain_partial:
    duration = end - start
    print(f"  {tier:<12} {name:<25} {start:>8.0f}ms → {end:>8.0f}ms  ({duration:.0f}ms)")
print(f"\n  用户看到首字时间: {t_partial_total:.0f}ms")

In [ ]:
# ============================================================
# CELL 5: 完整链路甘特图（主图）
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle("小采 1.0 全栈分布式 Trace\n从用户点击到看到第一个字",
             fontsize=17, fontweight='bold', y=0.99)

TIER_COLORS = {
    "UI 层":     "#3498db",
    "网络层":    "#e67e22",
    "API 网关":  "#f39c12",
    "AI Engine": "#e74c3c",
}
TIER_ORDER = ["UI 层", "网络层", "API 网关", "AI Engine"]

# ── 主图: 全链路甘特图 ──────────────────────────────────────
ax1 = axes[0, 0]
ax1.set_title("完整请求链路甘特图\n（用户点击 → 首字符出现）", fontweight='bold')

# 按层级绘制（每层一行）
tier_y = {tier: i for i, tier in enumerate(reversed(TIER_ORDER))}

for tier, name, start, end, color in chain_partial:
    y   = tier_y.get(tier, 0)
    dur = end - start
    ax1.barh(y, dur, left=start, height=0.7,
             color=color, alpha=0.85, edgecolor='white', linewidth=0.5)

    # 标注文字
    mid = start + dur / 2
    if dur > t_partial_total * 0.03:  # 只在足够宽时标文字
        ax1.text(mid, y, f"{name}\n{dur:.0f}ms",
                 ha='center', va='center', fontsize=7,
                 color='white', fontweight='bold')
    else:
        ax1.text(end + t_partial_total * 0.005, y, f"{name}",
                 va='center', fontsize=6, color='black')

# 标注关键时间点
ax1.axvline(x=t_partial_total, color='green', linestyle='--', linewidth=2,
            label=f'首字: {t_partial_total:.0f}ms')

ax1.set_yticks(list(tier_y.values()))
ax1.set_yticklabels(list(reversed(TIER_ORDER)), fontsize=10)
ax1.set_xlabel("时间 (ms, 从用户点击开始)", fontsize=10)
ax1.legend(fontsize=9, loc='lower right')
ax1.grid(True, axis='x', alpha=0.3)

# ── 子图 2: 各层时间占比 ──────────────────────────────────
ax2 = axes[0, 1]
ax2.set_title("各层时间占比", fontweight='bold')

tier_totals = {}
for tier, name, start, end, color in chain_partial:
    dur = end - start
    tier_totals[tier] = tier_totals.get(tier, 0) + dur

pie_labels = [f"{k}\n{v:.0f}ms" for k, v in tier_totals.items()]
pie_vals   = list(tier_totals.values())
pie_cols   = [TIER_COLORS[k] for k in tier_totals.keys()]

wedges, texts, autotexts = ax2.pie(
    pie_vals, labels=pie_labels, colors=pie_cols,
    autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 9}
)

# ── 子图 3: E2E 实测分布 ──────────────────────────────────
ax3 = axes[1, 0]
ax3.set_title("E2E 实测结果\n（首字符时间分布）", fontweight='bold')

if e2e_results:
    partial_firsts = [r["t_first_ms"] for r in e2e_partial if r.get("t_first_ms")]
    full_firsts    = [r["t_first_ms"] for r in e2e_full    if r.get("t_first_ms")]

    data_to_plot = []
    labels_bp = []
    colors_bp = []
    if partial_firsts:
        data_to_plot.append(partial_firsts)
        labels_bp.append(f"部分信息\n(n={len(partial_firsts)})")
        colors_bp.append("lightblue")
    if full_firsts:
        data_to_plot.append(full_firsts)
        labels_bp.append(f"完整信息\n(n={len(full_firsts)})")
        colors_bp.append("lightyellow")

    if data_to_plot:
        bp = ax3.boxplot(data_to_plot, labels=labels_bp, patch_artist=True,
                         boxprops=dict(facecolor='lightblue'),
                         medianprops=dict(color='red', linewidth=2))
        for patch, color in zip(bp['boxes'], colors_bp):
            patch.set_facecolor(color)
        ax3.set_ylabel("首字符时间 (ms)")
        ax3.grid(True, axis='y', alpha=0.3)

        # 添加均值标注
        for i, (data, label) in enumerate(zip(data_to_plot, labels_bp), 1):
            mean_v = np.mean(data)
            ax3.text(i, mean_v, f"  avg\n  {mean_v:.0f}ms",
                     va='center', fontsize=9, color='darkblue')
else:
    ax3.text(0.5, 0.5, "E2E 测量无数据",
             ha='center', va='center', transform=ax3.transAxes)

# ── 子图 4: 优化路线图 ──────────────────────────────────────
ax4 = axes[1, 1]
ax4.set_title("优化路线图\n（各层可优化空间）", fontweight='bold')
ax4.axis('off')

# 优化建议表格
optimizations = [
    ("P0 - 紧急",  "AI Engine",  "ENGINE-034: LLM failover 超时优化",   "最坏 264s → <30s"),
    ("P1 - 重要",  "AI Engine",  "Greeter+Req 并行化（非串行）",          f"-{baseline['agent_greeter_ms']:.0f}ms"),
    ("P1 - 重要",  "AI Engine",  "Embedding 结果 Redis 缓存",            "-80ms (cache HIT)"),
    ("P1 - 重要",  "AI Engine",  "Session 迁移至 Redis",                  "<1ms→持久化"),
    ("P2 - 改进",  "UI 层",      "消息列表虚拟化（react-window）",        "长对话 -50% 渲染"),
    ("P2 - 改进",  "网络层",     "HTTP/2 multiplexing",                  "并发请求 -30%"),
    ("P3 - 可选",  "API 网关",   "Edge Cache 静态资源 CDN",              "首屏 FCP -200ms"),
]

table_data = [[p, layer, opt, saving] for p, layer, opt, saving in optimizations]
table = ax4.table(
    cellText=table_data,
    colLabels=["优先级", "层级", "优化项", "预期收益"],
    cellLoc='left',
    loc='center',
    bbox=[0, 0, 1, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(8)

# 行颜色
row_colors = {
    "P0": "#ffcccc",
    "P1": "#ffe0cc",
    "P2": "#fff4cc",
    "P3": "#e8f5e9",
}
for i, (p, *_) in enumerate(optimizations, 1):
    color = row_colors.get(p[:2], "white")
    for j in range(4):
        table[i, j].set_facecolor(color)

plt.tight_layout(rect=[0, 0, 1, 0.97])
ts = datetime.now().strftime("%Y%m%d_%H%M")
save_path = f"{OUTPUT_DIR}/full-stack-distributed-trace_{ts}.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"图表已保存: {save_path}")

In [ ]:
# ============================================================
# CELL 6: 对比分析 — Cache HIT vs MISS 场景
# ============================================================

print("=" * 60)
print("场景对比: Cache HIT vs MISS")
print("=" * 60)

# 场景 A: 冷启动（Cache MISS，LLM Failover 正常）
# 场景 B: 热路径（Embedding Cache HIT，LLM 稳定）

def summarize_chain(name, greeter_ms, req_ms, clerk_ms, embed_cache_hit=False):
    embed_overhead = 5 if embed_cache_hit else baseline.get("agent_requirement_ms", 100) * 0.1

    t_ui   = baseline["ui_click_to_state_ms"] + baseline["ui_react_render_ms"]
    t_net  = baseline["net_tcp_ms"] + baseline["api_routing_ms"] + baseline["api_hop_ms"]
    t_llm  = greeter_ms + req_ms
    t_back = baseline["net_download_ms"] + baseline["ui_react_render_ms"] + baseline["ui_dom_paint_ms"]
    t_total_to_first = t_ui + t_net + t_llm + t_back

    print(f"\n  场景: {name}")
    print(f"    UI 层:          {t_ui:.0f}ms")
    print(f"    网络+API层:     {t_net:.0f}ms")
    print(f"    LLM 处理:       {t_llm:.0f}ms (Greeter {greeter_ms:.0f}ms + Req {req_ms:.0f}ms)")
    print(f"    回传+渲染:      {t_back:.0f}ms")
    print(f"    ---------------------------")
    print(f"    首字符时间:     {t_total_to_first:.0f}ms")
    print(f"    完整流程(+Doc): {t_total_to_first + clerk_ms:.0f}ms")
    return t_total_to_first

t_normal = summarize_chain(
    "正常情况 (APIPro, P50)",
    greeter_ms=baseline["agent_greeter_ms"],
    req_ms=baseline["agent_requirement_ms"],
    clerk_ms=baseline["agent_clerk_ms"]
)

t_fast = summarize_chain(
    "最快情况 (缓存命中, LLM 快速响应)",
    greeter_ms=baseline["agent_greeter_ms"] * 0.5,
    req_ms=baseline["agent_requirement_ms"] * 0.5,
    clerk_ms=baseline["agent_clerk_ms"] * 0.5,
    embed_cache_hit=True
)

t_slow = summarize_chain(
    "最慢情况 (APIPro 超时, 切 Ollama)",
    greeter_ms=30000,  # APIPro 30s 超时
    req_ms=60000,      # Ollama 60s timeout
    clerk_ms=60000
)

print(f"\n[关键对比]")
print(f"  正常场景首字时间: {t_normal:.0f}ms")
print(f"  最快场景首字时间: {t_fast:.0f}ms")
print(f"  最慢场景首字时间: {t_slow:.0f}ms (ENGINE-034 待修复)")
print(f"\n  最坏情况与最佳情况差距: {t_slow - t_fast:.0f}ms")
print(f"  优化 ENGINE-034 后最坏情况: ~30000ms (30s timeout + failover)")

In [ ]:
# ============================================================
# CELL 7: 最终汇总报告 + 保存
# ============================================================

print("=" * 70)
print("小采 1.0 全栈性能分析报告")
print("=" * 70)

print("""
一、架构概述
============
前端: React 18 + Vite + Ant Design (port 3001)
  └── EventSource API → /graphql/stream (GET)
API 网关: FastAPI + Strawberry GraphQL (port 8001)
  └── SSE 转发 → /api/v1/requirement/stream (POST)
AI Engine: FastAPI + LangGraph 状态机 (port 8002)
  ├── GreeterAgent: LLM 意图识别
  ├── RequirementAgent: 字段提取 (LLM + JSON)
  ├── category_agent: 品类识别 (关键词, ~5ms)
  └── ClerkAgent: PR 文档生成 (LLM)
数据层: PostgreSQL(session) + Redis(状态广播) + Qdrant(向量检索)
""")

print("二、关键性能指标")
print("=" * 50)
print(f"  首字符时间 (TTFC):  ~{t_normal:.0f}ms (正常情况)")
print(f"  E2E 完整响应:       ~{t_normal + baseline['agent_clerk_ms']:.0f}ms (含 PR 文档生成)")
print(f"  LLM 占比:           ~{(baseline['agent_greeter_ms'] + baseline['agent_requirement_ms']) / t_normal * 100:.0f}%")

print("\n三、各层延迟分布")
print("=" * 50)
for tier, name, start, end, color in chain_partial:
    dur = end - start
    pct = dur / t_partial_total * 100
    bar = "#" * int(pct / 2)
    print(f"  {tier:<12} {bar:<25} {pct:>5.1f}%  {dur:>7.0f}ms  {name}")

print("\n四、已知瓶颈与优化路线")
print("=" * 50)
print("  [P0] ENGINE-034: LLM failover 超时机制导致最坏情况 264s")
print("       → 修复方案: 减少每次 retry 超时, 并行多 provider")
print("  [P1] GreeterAgent + RequirementAgent 串行 LLM 调用")
print(f"       → 当前串行: {baseline['agent_greeter_ms']:.0f}ms + {baseline['agent_requirement_ms']:.0f}ms")
print("       → 优化方向: 合并 prompt 一次调用")
print("  [P1] Embedding 结果缓存 (Redis TTL 300s)")
print("       → 重复消息命中缓存可节省 ~80ms")
print("  [P2] 消息列表无虚拟化 (长对话性能退化)")
print("       → 考虑 react-window / react-virtualized")

print("\n五、盲区排查")
print("=" * 50)
print("  已覆盖:")
print("    UI 层: React 渲染 + EventSource 建连 + SSE 接收")
print("    网络层: TCP/TLS + TTFB + DNS")
print("    API 层: FastAPI路由 + CORS + GraphQL解析 + 内部hop")
print("    Agent层: Greeter + Requirement + Clerk + 品类识别")
print("    数据层: PostgreSQL + Redis + Qdrant + Embedding")
print("  潜在盲区:")
print("    - Session 持久化至 PostgreSQL 的延迟（当前内存）")
print("    - Qdrant 向量检索在 Agent 流程中的实际触发路径")
print("    - 生产环境 Nginx TLS 握手时间 (~20-50ms)")

# 保存完整报告
snapshot = {
    "timestamp":      datetime.now().isoformat(),
    "notebook":       "13-full-stack-distributed-trace",
    "baseline":       baseline,
    "chain_partial":  [
        {"tier": t[0], "name": t[1], "start_ms": t[2], "end_ms": t[3]}
        for t in chain_partial
    ],
    "t_first_char_ms": t_partial_total,
    "e2e_results":    e2e_results,
    "scenarios": {
        "normal_ttfc_ms": t_normal,
        "fast_ttfc_ms":   t_fast,
        "slow_ttfc_ms":   t_slow,
    },
    "optimizations": [
        {"priority": p, "layer": l, "item": o, "saving": s}
        for p, l, o, s in optimizations
    ],
}

ts = datetime.now().strftime("%Y%m%d_%H%M")
json_path = f"{OUTPUT_DIR}/full-stack-distributed-trace_{ts}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(snapshot, f, ensure_ascii=False, indent=2, default=str)

print(f"\n完整报告已保存: {json_path}")
print("\n13-full-stack-distributed-trace.ipynb 执行完成")
print("\n所有 Benchmark Notebooks (00-13) 已完成")